In [16]:
import os
from dotenv import load_dotenv

# 1. Load Environment Variables
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [12]:
loader = PyPDFLoader("datascience_and_analytics.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
final_chunks = text_splitter.split_documents(docs)

In [13]:

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    return HuggingFaceEmbeddings(model_name=model_name)

embedding = download_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7325.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
vectorstore = Chroma.from_documents(
documents=final_chunks,
embedding=embedding,
persist_directory="./rag_db"
)

In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [21]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)



In [22]:
system_prompt = (
    "You are a helpful assistant. Use the context provided to answer the user's question accurately. "
    "If the answer isn't in the context, say you don't know.\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [23]:
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [24]:
result = rag_chain.invoke({"input": "What is the summary of this document?"})
result

{'input': 'What is the summary of this document?',
 'context': [Document(metadata={'moddate': '2025-07-17T06:01:53+00:00', 'producer': 'iLovePDF', 'total_pages': 213, 'creator': 'PyPDF', 'creationdate': '', 'page_label': '12', 'source': 'datascience_and_analytics.pdf', 'page': 11}, page_content='Real-World Example\nIn an email, the sender, recipient, and timestamp are structured data, while the\nbody of the email is unstructured data.\nSummary\nStructured data is ideal for transactional and operational tasks due to its orga-\nnization and ease of analysis. Unstructured data, while harder to manage, holds\nvaluable insights—especially when analyzed with modern AI and machine learning\ntechniques. Most organizations today leverage both types to gain a comprehensive\nunderstanding of their operations and customer behavior.\n9'),
  Document(metadata={'moddate': '2025-07-17T06:01:53+00:00', 'page': 1, 'creationdate': '', 'source': 'datascience_and_analytics.pdf', 'creator': 'PyPDF', 'produc

In [25]:
print(f"Answer: {result['answer']}")

# Display Sources
for doc in result["context"]:
    print(f"• Page {doc.metadata.get('page')} from {doc.metadata.get('source')}")

Answer: Structured data is ideal for transactional and operational tasks because it is organized and easy to analyze. Unstructured data, though harder to manage, contains valuable insights, especially when analyzed with modern AI and machine learning techniques. Most organizations today use both types of data to get a complete understanding of their operations and customer behavior.
• Page 11 from datascience_and_analytics.pdf
• Page 1 from datascience_and_analytics.pdf
• Page 10 from datascience_and_analytics.pdf
